# Лабораторная работа №11: LLM агент и LangGraph

[Исходное задание](https://colab.research.google.com/drive/1KG4sK5pGyce6wvUGRU4TbOzm8aIeUIy2)

Выполнение через совместимый API RouterAI. API-ключ берётся из окружения или локального `.env`. Сохранённые результаты получены реальными запросами.



## Задание

##Задание на семинар
1. Используя агенты и инструмент @tool, реализуйте следующую систему. Есть список ключевых слов. Есть текст. БЯМ, используя цепочку,  определяет, содержатся ли в тексте все ключевые слова. Если содержатся, то модель на почту отправляет этот текст. Если не содержатся, то в интернете модель ищет статью со всеми ключевыми словами. Найденную статью модель отправляет на почту.
2. Реализуйте п.1 на LangGraph.

## Подключение

In [ ]:
from pathlib import Path
import sys

# При открытии из подпапки находим общий модуль конфигурации.
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "lab_runtime.py").exists():
        sys.path.insert(0, str(candidate))
        break

from lab_runtime import make_llm, json_result, ASSETS
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown, Image, Audio

llm = make_llm()
print("Модель:", llm.model_name)


## 1. Инструменты: проверка цепочкой LLM, поиск в интернете и письмо

Письмо сохраняется как настоящий файл `.eml`; внешняя доставка требует настройки SMTP. Поиск загружает страницу из интернета, модель анализирует текст.

In [ ]:
import json
import requests
import re
from email.message import EmailMessage
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

# Для выбора инструментов используем более сильную, но недорогую модель.
llm = make_llm(model="openai/gpt-4.1-mini", temperature=0, max_tokens=2000)

KEYWORDS = ["Python", "программирование", "язык"]
EMAIL_TO = "teacher@example.edu"
TEXT_OK = "Python — высокоуровневый язык программирования. Python применяется в анализе данных."
TEXT_MISSING = "Сегодня хорошая погода, можно гулять в парке."
outbox = []
check_chain = ChatPromptTemplate.from_messages([
    ("system", 'Проверь наличие всех ключевых слов, учитывая язык и перевод. Верни JSON {{"all_found":true/false,"missing":[]}}.'),
    ("human", "Слова: {keywords}\nТекст: {text}")
]) | llm | StrOutputParser()

@tool
def check_keywords(text: str, keywords: list[str]) -> dict:
    """Проверь с помощью LLM-цепочки, содержит ли текст все ключевые слова."""
    return json_result(check_chain.invoke({"text": text, "keywords": keywords}))

@tool
def search_article(keywords: list[str]) -> str:
    """Найди в интернете статью по ключевым словам через Wikipedia, верни текст со ссылкой."""
    query = " ".join(keywords)
    response = requests.get("https://ru.wikipedia.org/w/api.php", params={
        "action":"query", "format":"json", "generator":"search", "gsrsearch":query,
        "gsrlimit":1, "prop":"extracts", "explaintext":1, "exintro":1
    }, headers={"User-Agent":"UniversityCoursework/1.0"}, timeout=45)
    response.raise_for_status()
    pages = response.json().get("query", {}).get("pages", {})
    if not pages:
        raise RuntimeError("Статья по ключевым словам не найдена")
    page = next(iter(pages.values()))
    return f"{page['title']}\nhttps://ru.wikipedia.org/?curid={page['pageid']}\n{page['extract'][:1500]}"

@tool
def send_email(email: str, text: str) -> str:
    """Подготовь письмо с текстом и сохрани его в outbox в формате eml."""
    message = EmailMessage()
    message["To"] = email
    message["From"] = "student@example.edu"
    message["Subject"] = "Статья по ключевым словам"
    message.set_content(text)
    folder = ASSETS / "outbox"
    folder.mkdir(exist_ok=True)
    path = folder / f"letter_{len(outbox)+1}.eml"
    path.write_bytes(message.as_bytes())
    outbox.append({"email":email, "text":text, "path":str(path)})
    return f"Письмо подготовлено и сохранено в {path.name}. Внешняя доставка не выполнялась."

TOOLS = [check_keywords, search_article, send_email]
TOOLS_BY_NAME = {t.name:t for t in TOOLS}
llm_with_tools = llm.bind_tools(TOOLS)
print("Модель получила инструменты:", list(TOOLS_BY_NAME))


## 2. LLM самостоятельно выбирает tool_calls

In [ ]:
SYSTEM_PROMPT = f"""Ты агент. Проверь исходный текст инструментом check_keywords.
Если всех слов нет, найди статью search_article и проверь её тем же check_keywords.
Если все слова найдены, подготовь письмо send_email с исходным текстом или найденной статьёй.
Обязательно фактически вызови send_email, а не просто напиши, что собираешься это сделать.
Не возвращай финальный текст, пока send_email не выполнен. Не заявляй о доставке письма: send_email только сохраняет eml. После этого верни итог.
Ключевые слова: {KEYWORDS}. Получатель: {EMAIL_TO}."""

def run_llm_agent(text, max_steps=10):
    messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=text)]
    trace = []
    for _ in range(max_steps):
        # Здесь реальная LLM получает запрос, историю и схемы инструментов.
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            # Проверяем завершение задачи, не выбирая инструмент за модель.
            # При преждевременном завершении просим модель продолжить выполнение.
            if not any(step["tool"] == "send_email" for step in trace) or not response.content:
                messages.append(HumanMessage(content="Задача ещё не завершена. Выполни оставшиеся действия доступными инструментами и затем дай итоговый ответ."))
                continue
            return {"answer":response.content, "trace":trace, "messages":messages}
        for call in response.tool_calls:
            # Имя и аргументы выбраны моделью. Python только выполняет запрос.
            result = TOOLS_BY_NAME[call["name"]].invoke(call["args"])
            trace.append({"tool":call["name"], "args":call["args"], "result":result})
            messages.append(ToolMessage(content=json.dumps(result, ensure_ascii=False), tool_call_id=call["id"]))
        # Следующая итерация снова вызывает LLM уже с результатами tools.
    raise RuntimeError("Превышен лимит агентных шагов")

for text in [TEXT_OK, TEXT_MISSING]:
    result = run_llm_agent(text)
    print("\nТекст:", text)
    print("Выбранные моделью tools:", " → ".join(s["tool"] for s in result["trace"]))
    print("Итог LLM:", result["answer"])
    assert any(s["tool"] == "send_email" for s in result["trace"])


## 3. Тот же агент в LangGraph

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def agent_node(state):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, {"tools":"tools", "__end__":END})
builder.add_edge("tools", "agent")
agent_graph = builder.compile()

result = agent_graph.invoke({"messages":[SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=TEXT_MISSING)]}, config={"recursion_limit":25})
for message in result["messages"]:
    for call in getattr(message, "tool_calls", []):
        print("LLM →", call["name"], call["args"])
print("Итог:", result["messages"][-1].content)
print("Писем в outbox:", len(outbox))


## 4. Визуализация графа

In [ ]:
print(agent_graph.get_graph().draw_mermaid())
diagram = ASSETS / "lab11_langgraph.jpg"
display(Image(filename=str(diagram), width=850))


## Вывод

Реальная модель выбирает имя и аргументы инструментов через tool_calls. Результаты возвращаются через ToolMessage. LangGraph организует цикл agent → tools → agent; отсутствие tool_calls завершает работу. Проверка текста использует LLM-цепочку, поиск обращается к интернету, письмо сохраняется в eml.